# Exercise 1 — Thermostat Tuning in NVT Equilibration

**MARVEL–ICTP School 2026**

In this exercise you study how the Nosé–Hoover thermostat parameter `tdamp` controls the speed and quality of NVT equilibration for two water models: **SPC/Fw** (flexible) and **TIP3P** (rigid via SHAKE). You will:

1. Review the theory of the Nosé–Hoover thermostat and the physical meaning of `tdamp`
2. Choose three values of `tdamp` within the range [10, 1000] fs, making sure that one of them is `1000` fs; run all three for SPC/Fw, and run one of them for TIP3P
3. Compare temperature and total-energy trajectories across models and thermostat settings
4. Run one longer simulation at `tdamp = 1000` fs and use it to assess convergence


## 1. The NVT Ensemble and Thermostats

Standard molecular dynamics integrates Newton's equations of motion:

$$m_i \ddot{\mathbf{r}}_i = \mathbf{F}_i$$

This generates trajectories in the **microcanonical (NVE) ensemble**: $E$, $V$, and $N$ are conserved, but temperature fluctuates around a mean set by the initial kinetic energy. The instantaneous temperature is:

$$T(t) = \frac{2 K(t)}{N_f k_B}, \qquad K(t) = \frac{1}{2}\sum_i m_i v_i^2(t)$$

where $N_f$ is the number of degrees of freedom ($3N - 3$ for a system with constrained total momentum).

To sample the **canonical (NVT) ensemble** — controlling temperature rather than energy — the equations of motion must be extended so the system can exchange energy with a heat bath at target temperature $T_0$. The two physically correct methods used in this course are:

| Method | Ensemble | Character |
|--------|----------|-----------|
| **Nosé–Hoover** | exact NVT | deterministic; adds a feedback variable $\xi$ |
| **Langevin** | exact NVT | stochastic; adds random forces and friction |

Both sample the correct canonical distribution, but they perturb the dynamics differently. This exercise focuses on Nosé–Hoover, which is activated by `fix nvt` in LAMMPS.


## 2. The Nosé–Hoover Thermostat

The Nosé–Hoover thermostat extends the equations of motion with an extra degree of freedom $\xi$ that acts as a time-varying friction coefficient:

$$\dot{\mathbf{r}}_i = \frac{\mathbf{p}_i}{m_i}$$

$$\dot{\mathbf{p}}_i = \mathbf{F}_i - \xi\, \mathbf{p}_i$$

$$\dot{\xi} = \frac{1}{Q}\Bigl(\sum_i \frac{\mathbf{p}_i^2}{m_i} - N_f k_B T_0\Bigr) = \frac{2K - N_f k_B T_0}{Q}$$

where $Q$ is the **thermostat mass** (inertia of the extended variable).

### Feedback mechanism

The equation for $\dot{\xi}$ is a direct feedback loop on the kinetic energy:

- If $K > N_f k_B T_0 / 2$ (**too hot**): $\dot{\xi} > 0$, friction grows, particles slow down $\to$ $T$ decreases
- If $K < N_f k_B T_0 / 2$ (**too cold**): $\dot{\xi} < 0$, friction becomes negative, particles speed up $\to$ $T$ increases

The response is not instantaneous: $\xi$ itself has inertia $Q$ and evolves continuously, so the coupling is smooth.

### Conserved quantity

The Nosé–Hoover equations do **not** conserve the physical energy $E = K + V$. Instead they conserve the extended Hamiltonian:

$$\mathcal{H}_\text{NH} = K + V + \frac{Q\xi^2}{2} + N_f k_B T_0\, \eta, \qquad \dot{\eta} = \xi$$

In a correct implementation, $\mathcal{H}_\text{NH}$ is constant to machine precision. In this exercise we monitor $E = K + V$ (`TotEng` in LAMMPS), which is **not** conserved but should stabilise around a constant band once equilibrium is reached.


## 3. The Thermostat Mass $Q$ and `tdamp`

In LAMMPS, $Q$ is set through the parameter `tdamp` (written $\tau$ below):

$$\boxed{Q = N_f\, k_B\, T_0\, \tau^2}$$

The factor $\tau^2$ ensures that the **characteristic oscillation period** of the thermostat variable $\xi$ around its equilibrium value equals $\tau$, regardless of system size or temperature. To see this: linearising the Nosé–Hoover equations around equilibrium gives damped oscillations of the kinetic energy with angular frequency $\omega \approx 1/\tau$.

### Physical meaning of `tdamp`

Think of the thermostat as a pendulum: $\tau$ is the period. The system temperature will oscillate around $T_0$ with characteristic time $\tau$ before damping to equilibrium.

| `tdamp` | $Q$ | Thermostat character |
|---------|-----|----------------------|
| small | light | stiff: fast T correction, frequent kicks to momenta, velocity correlations disrupted |
| large | heavy | loose: slow T correction, closer to NVE behaviour, long equilibration |

### Practical guideline

For condensed-phase simulations a safe range is:

$$\tau \approx 50\text{–}500 \times \Delta t$$

With $\Delta t = 0.5$ fs this gives $\tau \approx 25$–250 fs. The default value `tdamp = 100.0 fs` (= $200\,\Delta t$) sits comfortably in this range and is the standard choice in the provided input files.

> **Note for production runs:** if you plan to extract dynamical observables such as the diffusion coefficient or the velocity autocorrelation function, `tdamp` must be large enough not to suppress velocity correlations. A thermostat that is too stiff biases transport properties even in thermostat-on runs.


## 4. Setting `tdamp` in LAMMPS

### SPC/Fw (flexible model)

```lammps
variable tdamp   index 100.0     # thermostat damping time [fs]
variable t_target equal 298.15   # target temperature [K]

fix thermostat all nvt temp ${t_target} ${t_target} ${tdamp}
```

### TIP3P (rigid model — SHAKE required)

```lammps
variable tdamp   index 100.0
variable t_target equal 298.15

fix thermostat all nvt temp ${t_target} ${t_target} ${tdamp}

# SHAKE enforces rigid bond lengths and H-O-H angle throughout dynamics.
# Without it, TIP3P would behave as a flexible model and the geometry
# would drift away from the equilibrium values at each timestep.
fix constraints all shake 1.0e-6 200 0 b 1 a 1
```

The `fix shake` syntax is: `shake tolerance max_iter N_printout bond_types angle_types`. Here `b 1` and `a 1` refer to bond type 1 (O–H) and angle type 1 (H–O–H).

### Overriding `tdamp` from the command line

Because `tdamp` is declared as a LAMMPS variable with `index`, it can be overridden at runtime:

```bash
lmp -in ../../in.spcfw.lammps -var tdamp 50.0
lmp -in ../../in.tip3p.lammps -var tdamp 50.0
```

The `-var name value` flag takes precedence over the `index` default in the file. This makes parameter sweeps easy: one input file, one number changed on the command line.


## 5. Part A — Instructions

### Part A — `tdamp` sweep

Choose **three values** of `tdamp` in the range [10, 1000] fs, making sure that **one of them is `1000` fs**. Run all three for SPC/Fw, then choose **one** of those three values and run only that one for TIP3P.

Fill in your chosen values in the **configuration cell** below, then follow the steps.

**Step 1 — build the initial configurations (once only)**

From the `exercise_1/` directory:

```bash
lmp -in in.build.spcfw.lammps   # run simulation, writes initial.spcfw.data
lmp -in in.build.tip3p.lammps   # run simulation, writes initial.tip3p.data
```

**Step 2 — run the tdamp sweep for SPC/Fw and one TIP3P case**

Choose your three `tdamp` values in the configuration cell below, making sure one of them is `1000` fs. Then create the folders and run the corresponding simulations.

Below is **one complete example** with `tdamp = 1000`:

```bash
# SPC/Fw example with tdamp = 1000
cd simulations   # enter simulations folder
mkdir spcfw_tdamp_1000   # create output folder
cd spcfw_tdamp_1000   # enter folder
lmp -in ../../in.spcfw.lammps -var tdamp 1000   # run simulation
cd ../..   # go back
```

Use the same pattern for the other two SPC/Fw runs, always naming the folders with the **actual tdamp value**: for example, `spcfw_tdamp_10`, `spcfw_tdamp_100`, `spcfw_tdamp_1000`.

For TIP3P, run only **one** of your three chosen `tdamp` values, again using the actual value in the folder name: for example, if you choose `tdamp = 1000`, the folder should be called `tip3p_tdamp_1000`.


In [ ]:
# ============================================================
# STUDENT CONFIGURATION — fill in your chosen values here
# ============================================================

# Part A: tdamp sweep
# Choose 3 values in the range [10, 1000] fs.
# One of the three values must be 1000 fs.
# Use integers — they will also be used as directory names.
TDAMP_A = None
TDAMP_B = None
TDAMP_C = None
TDAMP_TIP3P = None


In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.titlesize": 15,
    "figure.dpi": 130,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

TIMESTEP_FS  = 0.5
T_TARGET     = 298.15
ROLLING_FRAC = 0.08
ROOT = Path(".").resolve()

# --- Part A: optional tdamp sweep ---
TDAMP_VALUES = [v for v in (TDAMP_A, TDAMP_B, TDAMP_C) if v is not None]
PART_A_ACTIVE = (
    len(TDAMP_VALUES) == 3
    and TDAMP_TIP3P is not None
    and TDAMP_TIP3P in set(TDAMP_VALUES)
)
MODEL_TDAMP_VALUES = {
    "spcfw": TDAMP_VALUES if PART_A_ACTIVE else [],
    "tip3p": [TDAMP_TIP3P] if PART_A_ACTIVE else [],
}
MODEL_COLORS = {
    "spcfw": ["#d62728", "#1f77b4", "#2ca02c"][:len(MODEL_TDAMP_VALUES['spcfw'])],
    "tip3p": ["#d62728"] if PART_A_ACTIVE else [],
}

NSTEPS_LONG = globals().get("NSTEPS_LONG", None)

# --- Part B: optional longer run at tdamp = 1000 fs ---
PART_B_ACTIVE = NSTEPS_LONG is not None

print(f"Root: {ROOT}")
if PART_A_ACTIVE:
    print(f"SPC/Fw tdamp values: {MODEL_TDAMP_VALUES['spcfw']} fs")
    print(f"TIP3P tdamp value: {TDAMP_TIP3P} fs")
else:
    print("Part A not configured yet — set TDAMP_A, TDAMP_B, TDAMP_C, and TDAMP_TIP3P to enable it.")
if PART_B_ACTIVE:
    print(f"Long run: {NSTEPS_LONG} steps ({NSTEPS_LONG * TIMESTEP_FS / 1000:.1f} ps) at tdamp = 1000 fs")
else:
    print("Part B not configured yet — set NSTEPS_LONG to enable it.")

In [ ]:
def parse_lammps_log(log_path: Path) -> pd.DataFrame:
    with log_path.open(encoding="utf-8") as fh:
        lines = fh.readlines()
    blocks: list[pd.DataFrame] = []
    i = 0
    while i < len(lines):
        tokens = lines[i].split()
        if tokens and tokens[0] == "Step":
            columns = tokens
            data, j = [], i + 1
            while j < len(lines):
                row = lines[j].split()
                if not row:
                    break
                try:
                    values = [float(v) for v in row]
                except ValueError:
                    break
                if len(values) != len(columns):
                    break
                data.append(values)
                j += 1
            if data:
                blocks.append(pd.DataFrame(data, columns=columns))
            i = j
            continue
        i += 1
    if not blocks:
        raise ValueError(f"No thermo block in {log_path}")
    df = pd.concat(blocks, ignore_index=True)
    df = df.loc[~df["Step"].duplicated()].copy()
    df["time_ps"] = df["Step"] * TIMESTEP_FS / 1000.0
    return df


def add_rolling(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    window = max(5, int(round(len(df) * ROLLING_FRAC)))
    out = df.copy()
    for c in cols:
        out[f"{c}_roll"] = out[c].rolling(window=window, min_periods=1).mean()
    return out


def try_load(log_path: Path) -> pd.DataFrame | None:
    if not log_path.exists():
        return None
    df = parse_lammps_log(log_path)
    return add_rolling(df, ["Temp", "TotEng"])

In [ ]:
if not PART_A_ACTIVE:
    print("Part A skipped — set TDAMP_A, TDAMP_B, TDAMP_C, and TDAMP_TIP3P in the configuration cell to enable.")
else:
    # Load tdamp-sweep logs for both models
    tdamp_data: dict[str, dict] = {"spcfw": {}, "tip3p": {}}

    for model in ("spcfw", "tip3p"):
        for tdamp in MODEL_TDAMP_VALUES[model]:
            key = f"{tdamp} fs"
            dirname = f"{model}_tdamp_{tdamp}"
            path = ROOT / "simulations" / dirname / "log.lammps"
            df = try_load(path)
            if df is not None:
                tdamp_data[model][key] = df
                print(f"  [{model}] tdamp={tdamp} fs: {len(df)} points, "
                      f"up to {df['time_ps'].iloc[-1]:.1f} ps")
            else:
                print(f"  [{model}] tdamp={tdamp} fs: NOT FOUND ({path})")

    if not any(tdamp_data[m] for m in tdamp_data):
        raise FileNotFoundError("No tdamp-sweep logs found. Run the LAMMPS simulations first.")

In [ ]:
def tail_stats(df: pd.DataFrame, col: str, frac: float = 0.20) -> dict:
    n = max(5, int(round(len(df) * frac)))
    return {
        "mean": float(df[col].tail(n).mean()),
        "std":  float(df[col].tail(n).std()),
        "drift": float(df[col].tail(n).mean() - df[col].head(n).mean()),
    }

if not PART_A_ACTIVE:
    print("Part A skipped.")
else:
    rows = []
    for model in ("spcfw", "tip3p"):
        for key, df in tdamp_data[model].items():
            Ts = tail_stats(df, "Temp")
            Es = tail_stats(df, "TotEng")
            rows.append({
                "model": model,
                "tdamp [fs]": key,
                "duration [ps]": float(df["time_ps"].iloc[-1]),
                "T_tail [K]": Ts["mean"],
                "T_std [K]": Ts["std"],
                "ΔT head→tail [K]": Ts["drift"],
                "Etot_tail [kcal/mol]": Es["mean"],
                "ΔEtot head→tail [kcal/mol]": Es["drift"],
            })

    summary = pd.DataFrame(rows).set_index(["model", "tdamp [fs]"])
    display(summary.round(3))
    print("\nConvergence checks:")
    print(f"  T_tail should be close to {T_TARGET} K")
    print("  ΔT head→tail ≈ 0 → temperature plateau reached")
    print("  ΔEtot head→tail ≈ 0 → energy band stabilised")

## 6. Part A — Analysis: Temperature vs Time

The plots below show $T(t)$ for SPC/Fw (left) and TIP3P (right). SPC/Fw includes the three chosen `tdamp` values, while TIP3P includes only the single selected value. The thin line is the raw instantaneous temperature; the dashed line is a rolling mean over 8% of the data.

In [ ]:
if not PART_A_ACTIVE:
    print("Part A skipped.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
    model_labels = {"spcfw": "SPC/Fw (flexible)", "tip3p": "TIP3P (rigid / SHAKE)"}

    for ax, (model, label) in zip(axes, model_labels.items()):
        for tdamp, color in zip(MODEL_TDAMP_VALUES[model], MODEL_COLORS[model]):
            key = f"{tdamp} fs"
            if key not in tdamp_data[model]:
                continue
            df = tdamp_data[model][key]
            ax.plot(df["time_ps"], df["Temp"],
                    lw=0.7, alpha=0.25, color=color)
            ax.plot(df["time_ps"], df["Temp_roll"],
                    lw=2.0, ls="--", color=color,
                    label=rf"$\tau$ = {tdamp} fs")

        ax.axhline(T_TARGET, color="k", lw=1.0, ls=":", label=f"$T_0$ = {T_TARGET} K")
        ax.set_title(label)
        ax.set_xlabel("Time [ps]")
        ax.legend(frameon=False)

    axes[0].set_ylabel("$T$ [K]")
    fig.suptitle("Temperature vs time — tdamp comparison", y=1.01)
    fig.tight_layout()
    plt.show()

**Discussion — Thermostat speed and model comparison.**

1. Which of your three `tdamp` values equilibrates fastest? Does the stiffest thermostat show visible high-frequency oscillations in $T(t)$?

2. Do SPC/Fw and TIP3P equilibrate at the same rate for the same `tdamp`? If not, what physical difference might explain it? *(Hint: think about the number of degrees of freedom and the effect of SHAKE.)*

## 7. Part A — Analysis: Total Energy vs Time

In NVT, $E_\text{tot} = K + V$ is **not** conserved — the thermostat continuously exchanges energy with the system. However, once equilibrated, $E_\text{tot}$ should fluctuate around a stable mean with no systematic drift.

Note that the **absolute value** of $E_\text{tot}$ differs between SPC/Fw and TIP3P because the two force fields have different parameters and TIP3P is rigid (no bond/angle energy contributions to $V$). The comparison across models is therefore qualitative; what matters within each panel is the **stability** of the band.


In [ ]:
if not PART_A_ACTIVE:
    print("Part A skipped.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, (model, label) in zip(axes, model_labels.items()):
        for tdamp, color in zip(MODEL_TDAMP_VALUES[model], MODEL_COLORS[model]):
            key = f"{tdamp} fs"
            if key not in tdamp_data[model]:
                continue
            df = tdamp_data[model][key]
            ax.plot(df["time_ps"], df["TotEng"],
                    lw=0.7, alpha=0.25, color=color)
            ax.plot(df["time_ps"], df["TotEng_roll"],
                    lw=2.0, ls="--", color=color,
                    label=rf"$\tau$ = {tdamp} fs")

        ax.set_title(label)
        ax.set_xlabel("Time [ps]")
        ax.set_ylabel("$E_\\mathrm{tot}$ [kcal/mol]")
        ax.legend(frameon=False)

    fig.suptitle("Total energy vs time — tdamp comparison", y=1.01)
    fig.tight_layout()
    plt.show()

**Discussion — Energy stability.**

3. Does $E_\text{tot}$ settle at the same absolute value for all three `tdamp` settings within the same model? Should it? Does the absolute value differ between SPC/Fw and TIP3P, and why?

## 8. Part B — Instructions

Run **one longer simulation** with fixed `tdamp = 1000.0` fs. Choose a run length long enough to decide whether the trajectory has converged. Suggested range: 200 000–1 000 000 steps (100–500 ps at 0.5 fs/step).

Fill in your chosen `NSTEPS_LONG` value in the configuration cell below. Then create the folders and run the corresponding simulations.

Below is **one complete example** with `nsteps = 20000`:

```bash
# SPC/Fw example with tdamp = 1000 and nsteps = 20000
cd simulations   # enter simulations folder
mkdir spcfw_conv_long   # create output folder
cd spcfw_conv_long   # enter folder
lmp -in ../../in.spcfw.lammps -var tdamp 1000.0 -var nsteps 20000   # run simulation
cd ../..   # go back
```

Use the same pattern for the corresponding TIP3P run, replacing the folder name and input file.

Each run writes `log.lammps` and `traj.lammpstrj` in its directory.


In [ ]:
# ============================================================
# STUDENT CONFIGURATION — Part B
# ============================================================

# Part B: one longer run at fixed tdamp = 1000 fs
# Choose the number of timesteps for the longer run.
# 1 step = 0.5 fs → 200 000 steps = 100 ps, 1 000 000 steps = 500 ps
NSTEPS_LONG = None

## 9. Part B — Analysis: Convergence from a Longer Run

A large thermostat timescale can slow equilibration. Here you run **one longer trajectory** at fixed `tdamp = 1000 fs` for both models and use it to judge whether convergence has been reached on that timescale.

A run is considered converged when the rolling mean of both $T$ and $E_\text{tot}$ shows **no systematic drift** in the last 20% of the trajectory. Use the summary table below to quantify this.

In [ ]:
NSTEPS_LONG = globals().get("NSTEPS_LONG", None)
PART_B_ACTIVE = NSTEPS_LONG is not None

if not PART_B_ACTIVE:
    print("Part B skipped — set NSTEPS_LONG in the configuration cell to enable.")
else:
    conv_data: dict[str, dict] = {"spcfw": {}, "tip3p": {}}
    conv_labels = {
        "long": f"tdamp = 1000 fs ({NSTEPS_LONG} steps = {NSTEPS_LONG * TIMESTEP_FS / 1000:.1f} ps)",
    }
    conv_colors = {"long": "#9467bd"}

    for model in ("spcfw", "tip3p"):
        for tag in ("long",):
            path = ROOT / "simulations" / f"{model}_conv_{tag}" / "log.lammps"
            df = try_load(path)
            if df is not None:
                conv_data[model][tag] = df
                print(f"  [{model}] {tag}: {len(df)} points, "
                      f"up to {df['time_ps'].iloc[-1]:.1f} ps")
            else:
                print(f"  [{model}] {tag}: NOT FOUND ({path})")

    if not any(conv_data[m] for m in conv_data):
        raise FileNotFoundError("No convergence logs found. Run Part B simulations first.")

    rows = []
    for model in ("spcfw", "tip3p"):
        for tag, df in conv_data[model].items():
            Ts = tail_stats(df, "Temp")
            Es = tail_stats(df, "TotEng")
            rows.append({
                "model": model, "run": tag,
                "duration [ps]": float(df["time_ps"].iloc[-1]),
                "T_tail [K]": Ts["mean"], "T_std [K]": Ts["std"],
                "ΔT head→tail [K]": Ts["drift"],
                "ΔEtot head→tail [kcal/mol]": Es["drift"],
            })
    display(pd.DataFrame(rows).set_index(["model", "run"]).round(3))

In [ ]:
NSTEPS_LONG = globals().get("NSTEPS_LONG", None)
PART_B_ACTIVE = NSTEPS_LONG is not None

if not PART_B_ACTIVE:
    print("Part B skipped.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

    for ax, (model, model_label) in zip(axes, model_labels.items()):
        for tag, color in conv_colors.items():
            if tag not in conv_data[model]:
                continue
            df = conv_data[model][tag]
            ax.plot(df["time_ps"], df["Temp"],
                    lw=0.7, alpha=0.25, color=color)
            ax.plot(df["time_ps"], df["Temp_roll"],
                    lw=2.0, ls="--", color=color, label=conv_labels[tag])

        ax.axhline(T_TARGET, color="k", lw=1.0, ls=":")
        ax.set_title(model_label)
        ax.set_xlabel("Time [ps]")
        ax.legend(frameon=False)

    axes[0].set_ylabel("$T$ [K]")
    fig.suptitle(r"Temperature convergence — $\tau$ = 1000 fs", y=1.01)
    fig.tight_layout()
    plt.show()

In [ ]:
NSTEPS_LONG = globals().get("NSTEPS_LONG", None)
PART_B_ACTIVE = NSTEPS_LONG is not None

if not PART_B_ACTIVE:
    print("Part B skipped.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, (model, model_label) in zip(axes, model_labels.items()):
        for tag, color in conv_colors.items():
            if tag not in conv_data[model]:
                continue
            df = conv_data[model][tag]
            ax.plot(df["time_ps"], df["TotEng"],
                    lw=0.7, alpha=0.25, color=color)
            ax.plot(df["time_ps"], df["TotEng_roll"],
                    lw=2.0, ls="--", color=color, label=conv_labels[tag])

        ax.set_title(model_label)
        ax.set_xlabel("Time [ps]")
        ax.set_ylabel("$E_\\mathrm{tot}$ [kcal/mol]")
        ax.legend(frameon=False)

    fig.suptitle(r"Total energy convergence — $\tau$ = 1000 fs", y=1.01)
    fig.tight_layout()
    plt.show()

**Discussion — Convergence and practical choice.**

4. In the longer run at `tdamp = 1000 fs`, does each model appear converged, or is there still a visible drift in $T(t)$ or $E_\text{tot}(t)$ by the end of the trajectory?

5. Combining Part A with the longer `tdamp = 1000 fs` run, what value of `tdamp` would you say is sufficient to reach convergence without making equilibration unnecessarily slow? Justify your answer.